In [1]:
import pandas as pd
df = pd.read_csv("train_original.csv")
df.head(10)

,text,label
0,خیلی ناراحت شدم وقتی خبر بدی شنیدم,sadness
1,یک فرشته به خواب یکنفر میاد و بهش میگه خدا گفت...,neutral
2,ترسیدم چون صدای عجیبی شنیدم,fear
3,اونقدر حواسمون بود که بقیه ناراحت نشن که الان ...,sadness
4,خیلی راحته که کدوم جالش,neutral
5,باز خوبه لااقل ماسک و دستکش و شیلد اینا دارن ا...,happiness
6,دوستان توئیتری لطفاً هر کدومتون ۴۸ میلیارد دلا...,neutral
7,کجاییی ؟ چنیننبودنت عجیب است,fear
8,برخیز بیا به کوچه فریاد کنیم آلاله صفت که سینه...,anger
9,حالم بد شد چون بوی بدی احساس کردم,disgust


In [2]:
!pip install gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 53.2 MB/s  0:00:00 eta 0:00:01


In [7]:
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

class WikiSentences:
    def __init__(self, filepath):
        self.filepath = filepath

    def __iter__(self):
        with open(self.filepath, "r", encoding="utf-8") as f:
            for line in f:
                tokens = simple_preprocess(line.strip())
                if tokens:
                    yield tokens

# file in the same folder
sentences = WikiSentences("Persian-WikiText-2.txt")

# Initialize model
w2v_model = Word2Vec(
    vector_size=300,
    window=5,
    min_count=5,
    workers=4
)

# Build vocab
w2v_model.build_vocab(sentences)
print("Vocab size:", len(w2v_model.wv.key_to_index))

# Train model
w2v_model.train(
    sentences,
    total_examples=w2v_model.corpus_count,
    epochs=w2v_model.epochs
)

# Save model
w2v_model.save("wiki_fa_w2v.model")
print("✅ Model trained and saved as wiki_fa_w2v.model")


Vocab size: 49448
✅ Model trained and saved as wiki_fa_w2v.model


In [9]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

df["text"] = df["text"].fillna("").astype(str)

tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(df["text"])

word_index = tokenizer.word_index
print("تعداد کلمات دیتاست:", len(word_index))

embedding_dim = 300
num_words = min(20000, len(word_index)+1)

embedding_matrix = np.zeros((num_words, embedding_dim))
for word, i in word_index.items():
    if i >= num_words:
        continue
    if word in w2v_model.wv:   # ویکی‌پدیا Word2Vec
        embedding_matrix[i] = w2v_model.wv[word]

print("✅ embedding_matrix ساخته شد:", embedding_matrix.shape)


تعداد کلمات دیتاست: 78990
✅ embedding_matrix ساخته شد: (20000, 300)


In [10]:
# sequence
X_words = tokenizer.texts_to_sequences(df["text"])
X_words = pad_sequences(X_words, maxlen=50, padding="post")

print("شکل X_words:", X_words.shape)


شکل X_words: (50125, 50)


In [15]:
df = df.dropna(subset=["label"])

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(df["text"])

X_words = tokenizer.texts_to_sequences(df["text"])
X_words = pad_sequences(X_words, maxlen=50, padding="post")

from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df["label"])
y = to_categorical(y)

print("X_words shape:", X_words.shape)
print("y shape:", y.shape)



X_words shape: (49614, 50)
y shape: (49614, 7)


In [16]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense, Dropout

# ورودی کلمات
word_input = Input(shape=(50,), name="word_input")
word_embedding = Embedding(
    input_dim=num_words,
    output_dim=embedding_dim,
    weights=[embedding_matrix],
    input_length=50,
    trainable=True   
)(word_input)

# RNN
rnn_out = SimpleRNN(128, return_sequences=False)(word_embedding)
dropout = Dropout(0.5)(rnn_out)
output = Dense(y.shape[1], activation="softmax")(dropout)

# مدل
model = Model(inputs=word_input, outputs=output)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()


Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 word_input (InputLayer)     [(None, 50)]              0         
                                                                 
 embedding_1 (Embedding)     (None, 50, 300)           6000000   
                                                                 
 simple_rnn_1 (SimpleRNN)    (None, 128)               54912     
                                                                 
 dropout_1 (Dropout)         (None, 128)               0         
                                                                 
 dense_1 (Dense)             (None, 7)                 903       
                                                                 
Total params: 6055815 (23.10 MB)
Trainable params: 6055815 (23.10 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [17]:
from sklearn.utils import class_weight
class_weights = class_weight.compute_class_weight('balanced',
                                                  classes=np.unique(np.argmax(y, axis=1)),
                                                  y=np.argmax(y, axis=1))
class_weights = dict(enumerate(class_weights))

history = model.fit(X_words, y, validation_split=0.2,
                    epochs=30, batch_size=128,
                    class_weight=class_weights)


Epoch 1/30


'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)


  1/311 [..............................] - ETA: 6:19 - loss: 1.9976 - accuracy: 0.2188

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)


  4/311 [..............................] - ETA: 37s - loss: 1.9008 - accuracy: 0.2871 

...

268/311 [========================>.....] - ETA: 1s - loss: 0.3050 - accuracy: 0.8883

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)


311/311 [==============================] - ETA: 0s - loss: 0.3071 - accuracy: 0.8876

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)


311/311 [==============================] - 9s 29ms/step - loss: 0.3071 - accuracy: 0.8876 - val_loss: 2.4150 - val_accuracy: 0.4092


In [18]:
train_loss, train_acc = model.evaluate(X_words, y, verbose=0)
print(f"Train accuracy: {train_acc:.3f}, loss: {train_loss:.3f}")

Train accuracy: 0.822, loss: 0.677


In [21]:
import pandas as pd
from tensorflow.keras.preprocessing.sequence import pad_sequences

test = pd.read_csv("test_file.csv")

label_mapping = {
    'HAPPY': 'happiness',
    'SAD': 'sadness',
    'OTHER': 'neutral',
    'ANGRY': 'anger',
    'SURPRISE': 'surprize',
    'HATE': 'disgust',
    'FEAR': 'fear',
}

test["label"] = test["label"].map(label_mapping)
test = test.dropna(subset=["label"])
test["text"] = test["text"].fillna("").astype(str)

X_test_words = tokenizer.texts_to_sequences(test["text"])
X_test_words = pad_sequences(X_test_words, maxlen=50, padding="post")

test["label_encoded"] = label_encoder.transform(test["label"])
y_test = to_categorical(test["label_encoded"], num_classes=y.shape[1])

test_loss, test_acc = model.evaluate(X_test_words, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.3f}, loss: {test_loss:.3f}")


Test accuracy: 0.170, loss: 3.583
